# CSCI/MATH 485 Assignment
## Customer Churn Prediction with XGBoost
## Starter Notebook

This notebook is compatible with **Jupyter Notebook** and **Google Colab**.

This starter code is only to get you started. You can change any of the existing code here to complete all the tasks.

Complete all `TODO` sections. Make sure your final submission includes:
- data analysis,
- a tuned XGBoost model,
- your chosen main evaluation metric and justification,
- interpretation of top important features,
- and a final comparison with the baseline model.


## 1. Setup


In [49]:
# If you are using Google Colab, uncomment the next line if xgboost is not installed.
# !pip install xgboost


In [50]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier

## 2. Load the Dataset

**Instructions **
- Load the IBM Telco Customer Churn dataset.
- Display the first few rows.
- Confirm the dataset shape.
- If the url doesn't work for you, download the csv file from the Canvas Assignment#4

In [51]:
url = "https://raw.githubusercontent.com/plotly/datasets/master/telco-customer-churn-by-IBM.csv"
df = pd.read_csv(url)

# Display the first 5 rows of the dataset (done below)
print(df.head())

# TODO:
# Display the shape of the dataset
print("Shape of the dataset:", df.shape)


   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

## 3. Data Exploration

Complete the following:
- Print all column names
- Show data types
- Count missing values in each column
- Show the distribution of the target variable
- Write a short note: Is this a classification or regression problem? Why is this useful in business?


In [52]:
# TODO:
# Print column names
# Print data types
# Print missing values for each column
# Print value counts for the target column

print("\nTarget distribution:")
print(df['Churn'].value_counts())

# Extra: display other information of the dataset that you think can be useful
print("\nDataset Info:")

print(df.info())



Target distribution:
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contrac

**TODO (write your answer below):**

1. What kind of machine learning problem is this?
2. Why is churn prediction important in a business setting?

This is a classification problem because the target variable "Churn" is categorical (Yes/No). Churn prediction is important in a business setting because it helps companies identify customers who are likely to leave, allowing them to take proactive measures to retain those customers and reduce revenue loss.


## 4. Basic Cleaning

Complet the following:
- Identify whether there is an ID column that should be removed
- Convert the target column into binary form if needed
- Convert any numeric-looking columns stored as text into numeric values


In [53]:
# Make a copy so the original data remains unchanged
df_clean = df.copy()

# TODO:
# 1. Drop any unnecessary identifier column(s)
# 2. Convert the target column to 0/1
# 3. Convert any columns that should be numeric into numeric type
# 4. Handle invalid parsing if needed

# Example structure only:
# df_clean = df_clean.drop(columns=[...])
# df_clean["Churn"] = df_clean["Churn"].map({"Yes": 1, "No": 0})
# df_clean["TotalCharges"] = pd.to_numeric(df_clean["TotalCharges"], errors="coerce")

df_clean = df_clean.drop(columns=["customerID"])
df_clean["Churn"] = df_clean["Churn"].map({"Yes": 1, "No": 0})
df_clean["TotalCharges"] = pd.to_numeric(df_clean["TotalCharges"], errors="coerce") 

df_clean.head()


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


## 5. Define Features and Target

Complet the following:
- Define `X` and `y`
- Set the correct target column


In [54]:
# TODO:
# Replace with the correct target column name
target_col = "Churn"

X = df_clean.drop(columns=[target_col])

y = df_clean[target_col]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)


Feature matrix shape: (7043, 19)
Target shape: (7043,)


## 6. Identify Numeric and Categorical Features

Complet the following:
- Create a list of numeric columns
- Create a list of categorical columns


In [55]:
# Identify numeric and categorical feature columns
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "str"]).columns.tolist()

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numeric features:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical features:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


## 7. Train/Test Split

Complet the following:
- Split the dataset into training and testing sets
- Use stratification if appropriate


In [56]:
# TODO:
# Create train/test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)


X_train shape: (5634, 19)
X_test shape: (1409, 19)


## 8. Preprocessing Pipelines

Build preprocessing for:
- numeric features
- categorical features

Then combine them into a `ColumnTransformer`.


In [57]:
# Numeric preprocessing: impute missing values then scale for logistic regression
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing: impute then one-hot encode
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both pipelines into a single ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

## 9. Baseline Model: Logistic Regression

Complete the following:
- Train a Logistic Regression model as the baseline
- Generate predictions
- Evaluate using multiple metrics
- You may need to adjust `max_iter` if the model is not converging.


In [58]:
# Build and fit the baseline logistic regression pipeline
baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

baseline_model.fit(X_train, y_train)

# Generate predicted labels and probabilities
baseline_preds = baseline_model.predict(X_test)
baseline_probs = baseline_model.predict_proba(X_test)[:, 1]

In [59]:
# Compute baseline evaluation metrics
baseline_metrics = {
    "Accuracy":  accuracy_score(y_test, baseline_preds),
    "Precision": precision_score(y_test, baseline_preds),
    "Recall":    recall_score(y_test, baseline_preds),
    "F1-Score":  f1_score(y_test, baseline_preds),
    "ROC-AUC":   roc_auc_score(y_test, baseline_probs),
}

# Display as a formatted table
baseline_metrics_df = pd.DataFrame(
    baseline_metrics.items(),
    columns=["Metric", "Score"]
)
baseline_metrics_df["Score"] = baseline_metrics_df["Score"].map("{:.4f}".format)

print("=" * 40)
print("  Logistic Regression — Evaluation")
print("=" * 40)
print(baseline_metrics_df.to_string(index=False))
print("=" * 40)

  Logistic Regression — Evaluation
   Metric  Score
 Accuracy 0.8055
Precision 0.6572
   Recall 0.5588
 F1-Score 0.6040
  ROC-AUC 0.8419


In [60]:
# Classification report
print("Classification Report:")
print("-" * 55)
print(classification_report(y_test, baseline_preds, target_names=["No Churn", "Churn"]))

# Confusion matrix displayed as a labeled DataFrame
cm = confusion_matrix(y_test, baseline_preds)
cm_df = pd.DataFrame(
    cm,
    index=["Actual: No Churn", "Actual: Churn"],
    columns=["Pred: No Churn", "Pred: Churn"]
)
print("Confusion Matrix:")
print("-" * 40)
print(cm_df)

Classification Report:
-------------------------------------------------------
              precision    recall  f1-score   support

    No Churn       0.85      0.89      0.87      1035
       Churn       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409

Confusion Matrix:
----------------------------------------
                  Pred: No Churn  Pred: Churn
Actual: No Churn             926          109
Actual: Churn                165          209


## 10. Choose Your Main Evaluation Metric

Choose **one main metric** for this churn problem.

You must explain:
- which metric you chose,
- why it is appropriate,
- and why it is more informative than accuracy alone for this problem.


**TODO (write your answer below):**

I'd recommend Recall (also called sensitivity) as the main metric. Here's why:

- The cost of missing a churner is high. If the model fails to flag a customer who's about to leave, the company loses that revenue with no chance to intervene. A false alarm (predicting churn when the customer stays) is much cheaper — at worst you offer a loyal customer a retention discount.
- The dataset is imbalanced (~73% No / ~27% Yes). A model that just predicts "No Churn" for everyone would hit ~73% accuracy, which looks decent but catches zero churners. Accuracy rewards the majority class and hides poor performance on the class you actually care about.
- Recall directly measures what matters: of all customers who actually churn, what fraction did we catch? Maximizing recall minimizes the costly missed churners.

## 11. XGBoost Model

Complet the following:
- Build an XGBoost pipeline
- Tune at least 3 hyperparameters
- Use either `GridSearchCV` or your own tuning approach


In [61]:
xgb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        eval_metric="logloss",
        random_state=42
    ))
])

# Broader hyperparameter grid — scale_pos_weight handles class imbalance for recall
param_grid = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__max_depth": [3, 5, 7],
    "classifier__learning_rate": [0.01, 0.05, 0.1],
    "classifier__scale_pos_weight": [1, 2, 3],
}

param_grid

{'classifier__n_estimators': [100, 200, 300],
 'classifier__max_depth': [3, 5, 7],
 'classifier__learning_rate': [0.01, 0.05, 0.1],
 'classifier__scale_pos_weight': [1, 2, 3]}

In [62]:
# TODO:
# Run GridSearchCV (or perform manual tuning)
# Choose a scoring metric that matches your selected main evaluation metric

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring="recall",   # TODO: replace with your chosen scoring metric
    cv=3,
    n_jobs=-1,
    verbose=1
)

# TODO:
# Fit grid search on training data

# your code here
grid_search.fit(X_train, y_train)



Fitting 3 folds for each of 81 candidates, totalling 243 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'classifier__learning_rate': [0.01, 0.05, ...], 'classifier__max_depth': [3, 5, ...], 'classifier__n_estimators': [100, 200, ...], 'classifier__scale_pos_weight': [1, 2, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'recall'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messa

In [63]:
# TODO:
# Print the best hyperparameters
# Save the best model

# your code here
print("Best Hyperparameters:", grid_search.best_params_)

best_xgb_model = grid_search.best_estimator_





Best Hyperparameters: {'classifier__learning_rate': 0.01, 'classifier__max_depth': 3, 'classifier__n_estimators': 100, 'classifier__scale_pos_weight': 3}


## 12. Evaluate the Tuned XGBoost Model

Evaluate XGBoost using the same metrics as the baseline.


In [ ]:
# TODO:
# Generate predictions and probabilities using the best XGBoost model
# Suggested variable names:
# xgb_preds
# xgb_probs

# your code here

xgb_preds = best_xgb_model.predict(X_test)
xgb_probs = best_xgb_model.predict_proba(X_test)[:, 1]


In [65]:
# TODO:
# Compute and print:
# - Accuracy
# - Precision
# - Recall
# - F1-score
# - ROC-AUC

# your code here
xgb_metrics = {
    "Accuracy":  accuracy_score(y_test, xgb_preds),
    "Precision": precision_score(y_test, xgb_preds),
    "Recall":    recall_score(y_test, xgb_preds),
    "F1-Score":  f1_score(y_test, xgb_preds),
    "ROC-AUC":   roc_auc_score(y_test, xgb_probs),
}

print("XGBoost Metrics:")
for metric, score in xgb_metrics.items():
    print(f"{metric}: {score:.4f}")
    


XGBoost Metrics:
Accuracy: 0.7111
Precision: 0.4744
Recall: 0.8182
F1-Score: 0.6006
ROC-AUC: 0.8368


In [66]:
# Optional but helpful
# TODO:
# Print classification report and confusion matrix

# your code here
print("Classification Report:")
print("-" * 55)
print(classification_report(y_test, xgb_preds, target_names=["No Churn", "Churn"]))
print("Confusion Matrix:")
cm_xgb = confusion_matrix(y_test, xgb_preds)
cm_xgb_df = pd.DataFrame(
    cm_xgb,
    index=["Actual: No Churn", "Actual: Churn"],
    columns=["Pred: No Churn", "Pred: Churn"]
)
print("-" * 40)
print(cm_xgb_df)    


Classification Report:
-------------------------------------------------------
              precision    recall  f1-score   support

    No Churn       0.91      0.67      0.77      1035
       Churn       0.47      0.82      0.60       374

    accuracy                           0.71      1409
   macro avg       0.69      0.75      0.69      1409
weighted avg       0.80      0.71      0.73      1409

Confusion Matrix:
----------------------------------------
                  Pred: No Churn  Pred: Churn
Actual: No Churn             696          339
Actual: Churn                 68          306


## 13. Feature Importance

Use the trained XGBoost model to:
- extract feature importances,
- recover transformed feature names,
- display the top 5 to 10 most important features.


In [67]:
# TODO:
# Access the fitted preprocessor and fitted XGBoost classifier from the pipeline

# Example structure:
# fitted_preprocessor = best_model.named_steps["preprocessor"]
# fitted_xgb = best_model.named_steps["classifier"]

# your code here
fitted_preprocessor = best_xgb_model.named_steps["preprocessor"]
fitted_xgb = best_xgb_model.named_steps["classifier"]



In [68]:
# Get transformed feature names from the preprocessor
feature_names = fitted_preprocessor.get_feature_names_out()

# Get feature importances from the XGBoost classifier
importances = fitted_xgb.feature_importances_

# Build a sorted DataFrame
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False).reset_index(drop=True)

In [69]:
# Display the top 10 most important features
print("=" * 45)
print("  Top 10 Features by Importance")
print("=" * 45)
print(importance_df.head(10).to_string(index=False))
print("=" * 45)

  Top 10 Features by Importance
                            Feature  Importance
       cat__Contract_Month-to-month    0.562186
   cat__InternetService_Fiber optic    0.092037
                cat__TechSupport_No    0.087252
             cat__OnlineSecurity_No    0.063623
                        num__tenure    0.044340
           cat__StreamingMovies_Yes    0.034491
cat__PaymentMethod_Electronic check    0.031944
                num__MonthlyCharges    0.030766
             cat__Contract_One year    0.026209
               cat__OnlineBackup_No    0.018759


## 14. Interpret the Top Features

Write a short interpretation of the most important features.

Your explanation should:
- use plain language,
- connect features to churn behavior,
- and explain what the company might learn from them.


**Feature Interpretation:**

- **Contract: Month-to-month (0.56)**: By far the most important feature. Month-to-month customers churn at much higher rates since there's no commitment keeping them. Incentivizing longer contracts (e.g., discounts for annual plans) could significantly reduce churn.
- **InternetService: Fiber optic (0.09)**: Fiber optic customers churn more, possibly due to higher costs or service quality issues specific to that tier. The company should investigate whether fiber optic customers feel they're getting value for their premium.
- **TechSupport: No (0.09)**: Customers without tech support churn more. Bundling tech support into plans — or offering it free for the first few months — could increase stickiness.
- **OnlineSecurity: No (0.06)**: Similar to tech support, lacking online security is associated with churn. These add-on services act as retention anchors.
- **tenure (0.04)**: Shorter-tenure customers are more likely to churn — they haven't yet built loyalty or dependence on the service. Retention efforts should target newer customers.
- **StreamingMovies: Yes (0.03)**: Customers with streaming services still churn, possibly because they're paying more overall and are more price-sensitive.
- **PaymentMethod: Electronic check (0.03)**: Electronic check users churn more often, possibly indicating less-engaged customers who haven't set up automatic payments. Nudging them toward auto-pay could improve retention.
- **MonthlyCharges (0.03)**: Higher monthly charges correlate with churn — customers paying more may feel they aren't getting enough value.
- **Contract: One year (0.03)**: One-year contracts provide moderate protection against churn, but less than two-year contracts.
- **OnlineBackup: No (0.02)**: Another missing add-on associated with churn — reinforcing the pattern that customers with fewer services are less "locked in."

## 15. Final Comparison: Logistic Regression vs XGBoost

Compare the two models using your results.

Your discussion should answer:
- Which model performed better?
- On which metric(s)?
- Why might XGBoost perform better on this dataset?
- What is one limitation of XGBoost?


In [70]:
# Side-by-side comparison of Logistic Regression vs XGBoost
comparison_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"],
    "Logistic Regression": [
        accuracy_score(y_test, baseline_preds),
        precision_score(y_test, baseline_preds),
        recall_score(y_test, baseline_preds),
        f1_score(y_test, baseline_preds),
        roc_auc_score(y_test, baseline_probs),
    ],
    "XGBoost": [
        accuracy_score(y_test, xgb_preds),
        precision_score(y_test, xgb_preds),
        recall_score(y_test, xgb_preds),
        f1_score(y_test, xgb_preds),
        roc_auc_score(y_test, xgb_probs),
    ],
})

# Format scores to 4 decimal places
comparison_df["Logistic Regression"] = comparison_df["Logistic Regression"].map("{:.4f}".format)
comparison_df["XGBoost"] = comparison_df["XGBoost"].map("{:.4f}".format)

print("=" * 55)
print("  Model Comparison: Logistic Regression vs XGBoost")
print("=" * 55)
print(comparison_df.to_string(index=False))
print("=" * 55)

  Model Comparison: Logistic Regression vs XGBoost
   Metric Logistic Regression XGBoost
 Accuracy              0.8055  0.7111
Precision              0.6572  0.4744
   Recall              0.5588  0.8182
 F1-Score              0.6040  0.6006
  ROC-AUC              0.8419  0.8368


**Final Comparison:**

- **Which model performed better?** XGBoost performed better on our chosen metric — recall — by a large margin (0.82 vs 0.56), meaning it catches 306 out of 374 churners compared to only 209 for logistic regression. Logistic regression had higher accuracy (0.81 vs 0.71) and precision (0.66 vs 0.47), but those gains come from conservatively predicting "No Churn" and missing many actual churners.

- **On which metric(s)?** XGBoost excelled on recall (0.82 vs 0.56). F1-score (0.60 vs 0.60) and ROC-AUC (0.84 vs 0.84) were nearly identical between the two models, indicating similar overall discriminative ability — the difference is in how aggressively XGBoost flags potential churners thanks to the `scale_pos_weight` parameter handling the class imbalance.

- **Why might XGBoost perform better?** XGBoost can capture non-linear relationships and feature interactions (e.g., short tenure *combined with* month-to-month contract) that logistic regression's linear decision boundary cannot model. The `scale_pos_weight` parameter also gave XGBoost a direct mechanism to prioritize the minority class, which is why recall improved so dramatically while ROC-AUC stayed comparable.

- **What is one limitation of XGBoost?** XGBoost is less interpretable than logistic regression. Logistic regression gives clear, signed coefficients showing each feature's direction and magnitude of effect, while XGBoost's feature importances only show relative contribution without indicating directionality. XGBoost also has more hyperparameters to tune, making it more prone to overfitting if not carefully validated.

## 16. Suggested Submission Checklist

Before submitting, make sure your notebook includes:
- completed code cells,
- outputs for each section,
- your selected main evaluation metric and justification,
- tuned XGBoost hyperparameters,
- feature importance results,
- interpretation of important features,
- and a final comparison of the two models.
